# Developmental LM — Phase 1 training

This notebook trains only the Phase 1 causal GRU next-phoneme model. Timing is fixed at 10 ms/frame, 5 frames/phoneme, and 1 frame overlap. Real training requires a North American English IPA-CHILDES export, a complete IPA feature table, and an empirical utterance-pause file.

## 1. Runtime setup
In Colab, select **Runtime → Change runtime type → T4 GPU** if desired. The baseline also runs on CPU, although the current minimal trainer does not yet include explicit device placement.

In [ ]:
import os, pathlib, subprocess, sys

REPO_URL = "https://github.com/ss-sebastian/developmental_checkpoints_word_recognition.git"
PROJECT = pathlib.Path("/content/developmental_checkpoints_word_recognition")
if PROJECT.exists():
    subprocess.run(["git", "-C", str(PROJECT), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{PROJECT}[panphon]", "pytest"], check=True)
os.chdir(PROJECT)
print("Project ready:", PROJECT)

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"], check=True)

## 2. Optional smoke training
This uses only the repository's synthetic test fixtures and verifies the training/checkpoint path. It is not scientific training data.

In [ ]:
RUN_SMOKE = True
if RUN_SMOKE:
    subprocess.run([sys.executable, "-m", "devlm.cli", "--config", "configs/smoke.toml"], check=True)

## 3. Mount Drive and set real-data paths
Edit the four paths below. No empirical phoneme-duration file is needed. Checkpoints should point to Drive so they survive Colab runtime resets.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATASET_PATH = "/content/drive/MyDrive/devlm/north_american_ipa_childes.jsonl"
FEATURE_TABLE_PATH = "/content/drive/MyDrive/devlm/ipa_feature_mapping.json"
PAUSE_DURATIONS_PATH = "/content/drive/MyDrive/devlm/empirical_utterance_pauses.json"
OUTPUT_DIR = "/content/drive/MyDrive/devlm/outputs/phase1"

for path in (DATASET_PATH, FEATURE_TABLE_PATH, PAUSE_DURATIONS_PATH):
    if not pathlib.Path(path).is_file():
        raise FileNotFoundError(f"Edit the notebook path; file not found: {path}")
pathlib.Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

## 4. Write the real-training configuration
The five-value envelope is symmetric and normalized by the implementation to peak 1.0. Phoneme extent and overlap remain fixed and are not configurable.

In [ ]:
import json

def toml_string(value):
    return json.dumps(str(value))

config_text = f'''[phase1]
dataset_path = {toml_string(DATASET_PATH)}
feature_table_path = {toml_string(FEATURE_TABLE_PATH)}
pause_durations_path = {toml_string(PAUSE_DURATIONS_PATH)}
output_dir = {toml_string(OUTPUT_DIR)}
seed = 20260822
validation_fraction = 0.1
noise_sigma = 0.05
phoneme_envelope = [0.3333333333, 0.6666666667, 1.0, 0.6666666667, 0.3333333333]
hidden_size = 128
num_layers = 1
dropout = 0.0
learning_rate = 0.001
gradient_clip_norm = 1.0
target_checkpoint_count = 30
'''
COLAB_CONFIG = pathlib.Path("/content/phase1_colab.toml")
COLAB_CONFIG.write_text(config_text, encoding="utf-8")
print(config_text)

## 5. Train
The command performs one developmental pass and writes periodic checkpoints plus validation metrics to `OUTPUT_DIR`.

In [ ]:
subprocess.run([sys.executable, "-m", "devlm.cli", "--config", str(COLAB_CONFIG)], check=True)